# 01 Extract — 小说原文 → 多轮对话数据

TaleTalk 流水线第一阶段：把一本小说的 txt 抽成「角色 + 台词」原始 jsonl，再切成多轮 ShareGPT SFT 数据，供 `02_train.ipynb` 训练 LoRA 使用。

**这一步只依赖 OpenAI 兼容的 LLM API**，可以是：

- 云端 API：DeepSeek / OpenAI / SiliconFlow / Moonshot / 阶跃 / 任意 OpenAI 兼容服务（填 .env 即可）
- 本地 vLLM / LLaMA Factory 起的 OpenAI server（同样走 .env 的 CUSTOM_*）

抽取支持断点续跑：相同 chunk 已抽过会从本地 cache 直接复用，重跑不会重复花钱。

## 0. 参数区

改完这个 cell 后**从头按顺序跑**即可。每次抽不同小说/不同角色就改这里。

In [ ]:
from pathlib import Path

# ====== 输入 ======
NOVEL_TXT = Path('novels/西游记白话文.txt')              # 小说原文 txt 路径（相对仓库根）
TARGET_ROLE = '孙悟空'                                        # 想复刻的角色名
NOVEL_TITLE = '西游记'                                         # 仅用于 system prompt 文案
RUN_NAME = 'xyj_wukong'                                        # 数据集前缀，决定输出文件名

# ====== 模型（抽取 + 训练 + 推理共用）======
# 三个 notebook (01/02/03) 共享同一份模型表。换模型只改 MODEL_CHOICE 一处。
# Qwen3 / Qwen3.5 / Qwen3.6 都是 instruction-tuned，可直接用于抽取（识别对话）+ 训练 LoRA。
MODEL_IDS = {
    'qwen3_5_9b':       'Qwen/Qwen3.5-9B',         # 9B, 推荐: W7900D 48GB 单卡跑得动
    'qwen3_5_27b':      'Qwen/Qwen3.5-27B',        # 27B, 需要 >= 64GB 显存或量化
    'qwen3_6_35b_a3b':  'Qwen/Qwen3.6-35B-A3B',    # 35B MoE A3B, 激活 3B, 性能/显存平衡
    'qwen3_8b':         'Qwen/Qwen3-8B',           # 备选: 更小更快
}
MODEL_CHOICE = 'qwen3_5_9b'
MODEL_ID = MODEL_IDS[MODEL_CHOICE]

# ====== 抽取后端 ======
# 'cloud_api'   — 用云端 LLM API（DeepSeek/OpenAI/SiliconFlow/Moonshot/阶跃等），跑下面 §2A
# 'local_model' — 在本机/云端 GPU 上起 vLLM + LLaMA Factory api server 抽取，跑下面 §2B
EXTRACTION_BACKEND = 'local_model'

# ====== 仅 cloud_api 时关心 ======
LLM_PLATFORM = 'custom'   # deepseek / openai / siliconflow / moonshot / custom

# ====== 仅 local_model 时关心 ======
# 默认复用上面的 MODEL_ID。如果想用别的更小/更专门的 instruct 模型抽取，改这里覆盖：
# LOCAL_MODEL_ID_OVERRIDE = 'Qwen/Qwen2.5-7B-Instruct'
LOCAL_MODEL_ID_OVERRIDE = ''   # 留空 = 复用 MODEL_ID
LOCAL_MODEL_PORT = 8000

# ====== 抽取行为 ======
MAX_WORKERS = 6              # 并发线程数
CHUNK_SIZE_TOKENS = 1500     # 单 chunk 喂给 LLM 的 token 数

# ====== 多轮 SFT 转换 ======
VALID_RATIO = 0.05
MAX_CONVERSATIONS = 0
SEED = 42

# ====== 路径 ======
REPO_DIR = Path.cwd().resolve()
while REPO_DIR != REPO_DIR.parent and not (REPO_DIR / 'extract').is_dir():
    REPO_DIR = REPO_DIR.parent
assert (REPO_DIR / 'extract').is_dir(), f'未找到 taletalk 仓库根目录: {REPO_DIR}'

# 相对路径以仓库根为基准
if not NOVEL_TXT.is_absolute():
    NOVEL_TXT = (REPO_DIR / NOVEL_TXT).resolve()

RAW_DIR = REPO_DIR / 'data' / 'raw'
SFT_DIR = REPO_DIR / 'data'
CACHE_DIR = REPO_DIR / 'cache'
for d in [RAW_DIR, SFT_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

RAW_JSONL = RAW_DIR / f'{RUN_NAME}_dialogues.jsonl'
TRAIN_JSON = SFT_DIR / f'{RUN_NAME}_chat_train.json'
VALID_JSON = SFT_DIR / f'{RUN_NAME}_chat_valid.json'

# 真正用于本地抽取的模型 id
LOCAL_MODEL_ID = LOCAL_MODEL_ID_OVERRIDE.strip() or MODEL_ID

print('repo:', REPO_DIR)
print('novel:', NOVEL_TXT, 'exists=', NOVEL_TXT.exists())
print('model:', MODEL_ID)
print('extraction backend:', EXTRACTION_BACKEND)
if EXTRACTION_BACKEND == 'local_model':
    print('local extraction model:', LOCAL_MODEL_ID)
print('run name:', RUN_NAME)

## 1. 安装抽取依赖

只装抽取阶段需要的几个小包，不会污染训练镜像。

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'requirements-extract.txt')])
print('extract deps installed')

## 2A. （可选 A）云端 API 抽取

**只在 `EXTRACTION_BACKEND = 'cloud_api'` 时跑这个 cell**。

两种填法任选其一：直接在下面写 `API_KEY` / `BASE_URL` / `MODEL_NAME`；或者在仓库根目录 `cp .env.example .env` 填好后留空这三个变量。

In [ ]:
if EXTRACTION_BACKEND != 'cloud_api':
    print(f'skip (EXTRACTION_BACKEND={EXTRACTION_BACKEND}, 跳过云端 API 配置)')
else:
    import os
    from dotenv import load_dotenv

    # ====== 直接填这三个（留空就走 .env 文件）======
    API_KEY = ''            # 比如 'sk-xxx' / 'EMPTY'（本地 vLLM 不校验 key 时填 EMPTY）
    BASE_URL = ''           # 比如 'http://localhost:8000/v1' / 'https://api.deepseek.com'
    MODEL_NAME = ''         # 比如 'Qwen/Qwen3-30B-A3B-Instruct-2507' / 'deepseek-chat' / 'step-3.7-flash'

    # 先尝试加载 .env（如果有），notebook 里填的值随后覆盖
    ENV_PATH = REPO_DIR / '.env'
    if ENV_PATH.exists():
        load_dotenv(ENV_PATH, override=False)
        print(f'loaded {ENV_PATH}')
    else:
        print(f'no .env file at {ENV_PATH}, 将使用 notebook 里填的值')

    PLATFORM_VARS = {
        'deepseek':    ('DEEPSEEK_API',        'DEEPSEEK_BASE_URL'),
        'openai':      ('OPENAI_API_KEY',      'OPENAI_BASE_URL'),
        'siliconflow': ('SILICONFLOW_API_KEY', 'SILICONFLOW_BASE_URL'),
        'moonshot':    ('MOONSHOT_API_KEY',    'MOONSHOT_BASE_URL'),
        'custom':      ('CUSTOM_API_KEY',      'CUSTOM_BASE_URL'),
    }
    key_var, url_var = PLATFORM_VARS[LLM_PLATFORM]

    # notebook 直填值覆盖环境变量
    if API_KEY.strip():
        os.environ[key_var] = API_KEY.strip()
    if BASE_URL.strip():
        os.environ[url_var] = BASE_URL.strip()
    if MODEL_NAME.strip():
        # extract.Config 通过 {PLATFORM}_MODEL_NAME 环境变量读 model name
        model_env = f'{LLM_PLATFORM.upper()}_MODEL_NAME'
        os.environ[model_env] = MODEL_NAME.strip()

    final_key = os.environ.get(key_var, "").strip()
    final_url = os.environ.get(url_var, "").strip()
    assert final_key, f'{key_var} 没有配置（notebook 里填 API_KEY 或在 .env 里设置）'
    assert final_url, f'{url_var} 没有配置（notebook 里填 BASE_URL 或在 .env 里设置）'

    key_disp = f'***{final_key[-4:]}' if len(final_key) > 4 else '***'
    print(f'platform = {LLM_PLATFORM}')
    print(f'base_url = {final_url}')
    print(f'api_key  = {key_disp}')
    if MODEL_NAME.strip():
        print(f'model    = {MODEL_NAME.strip()}')

    # 让后续 cell 的 Config 类读到正确平台
    os.environ['LLM_PLATFORM'] = LLM_PLATFORM

## 2B. （可选 B）本地模型抽取

**只在 `EXTRACTION_BACKEND = 'local_model'` 时跑下面三个 cell**。

流程：
1. 检查 / 自动安装 ROCm 版 vLLM（从 `https://wheels.vllm.ai/rocm/` 拉预编译 wheel，需要 Python 3.12 + ROCm 6.3+）
2. 用 ModelScope 把 `LOCAL_MODEL_ID` 下载到 `/network-workspace/models/`（已下载会跳过）
3. 后台起 `llamafactory-cli api`（vLLM 后端）+ 等就绪 + 自动配置 BASE_URL / MODEL_NAME

> vLLM 首次加载模型 + warmup 1-3 分钟，耐心等。  
> 关掉 kernel 会自动停掉这个 server。  
> 若 vLLM 装不上，可临时把 cell 里 `infer_backend: vllm` 改成 `infer_backend: huggingface` 降级（速度慢 3-5 倍但能跑）。

In [ ]:
if EXTRACTION_BACKEND == 'local_model':
    import sys, subprocess
    try:
        import vllm
        print(f'vllm already installed: {vllm.__version__}')
    except ImportError:
        py = sys.version_info
        if (py.major, py.minor) != (3, 12):
            raise RuntimeError(
                f'vLLM ROCm 预编译 wheel 仅支持 Python 3.12，当前是 {py.major}.{py.minor}。'
                ' 见 https://docs.vllm.ai/en/stable/getting_started/installation/gpu/rocm/'
            )
        # 镜像里应该自带 ROCm PyTorch，vLLM wheel 会校验兼容性
        # 用 pip 必须指定确切版本，先自动探测最新 ROCm variant 和 version
        print('detecting latest vllm rocm wheel ...')
        idx = subprocess.check_output(
            ['curl', '-s', 'https://wheels.vllm.ai/rocm/vllm'],
            text=True, timeout=60,
        )
        import re
        variants = re.findall(r'rocm(\d+)', idx)
        versions = re.findall(r'vllm-([\d.]+)', idx)
        assert variants and versions, f'解析 wheels.vllm.ai 失败: {idx[:500]}'
        variant = f'rocm{variants[0]}'
        version = versions[0].rstrip('.')
        wheel_url = f'https://wheels.vllm.ai/rocm/{version}/{variant}'
        spec = f'vllm=={version}+{variant}'
        print(f'installing {spec} from {wheel_url}')
        subprocess.check_call([
            sys.executable, '-m', 'pip', 'install', spec,
            '--extra-index-url', wheel_url,
        ], timeout=1800)
        import vllm
        print(f'vllm installed: {vllm.__version__}')
else:
    print('skip (using cloud_api)')

In [ ]:
if EXTRACTION_BACKEND == 'local_model':
    from modelscope import snapshot_download
    MODEL_CACHE_DIR = Path('/network-workspace/models')
    MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    print(f'downloading {LOCAL_MODEL_ID} to {MODEL_CACHE_DIR} ...')
    LOCAL_MODEL_DIR = Path(snapshot_download(LOCAL_MODEL_ID, cache_dir=str(MODEL_CACHE_DIR))).resolve()
    assert (LOCAL_MODEL_DIR / 'config.json').exists(), f'模型目录不完整: {LOCAL_MODEL_DIR}'
    print('model_dir:', LOCAL_MODEL_DIR)
else:
    print('skip (using cloud_api)')

In [ ]:
if EXTRACTION_BACKEND == 'local_model':
    import os, subprocess, time, atexit, signal, socket, urllib.request

    def _port_open(port):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.settimeout(0.3)
            return s.connect_ex(('127.0.0.1', port)) == 0

    # Qwen3.5 / Qwen3.6 是 vision-language 模型 (官方: "Towards Native Multimodal Agents")，
    # 没有 text-only 变体。vLLM 启动时会对 vision encoder 做 profiling, 在 ROCm 上经常挂住。
    # 官方文档 (vllm.docs/configuration/conserving_memory.md) 推荐的 text-only 用法：
    # 把 image / video 的 limit 设成 0, 跳过 vision profiling。
    # LLaMA Factory 的 vllm 封装不暴露这个 flag, 所以直接调 'vllm serve'。
    if _port_open(LOCAL_MODEL_PORT):
        print(f'port {LOCAL_MODEL_PORT} already in use — 直接复用现有 server')
    else:
        env = os.environ.copy()
        env['CUDA_VISIBLE_DEVICES'] = '0'
        env['HIP_VISIBLE_DEVICES'] = '0'
        env.setdefault('VLLM_USE_TRITON_FLASH_ATTN', '0')
        env.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')

        cmd = [
            'vllm', 'serve', str(LOCAL_MODEL_DIR),
            '--port', str(LOCAL_MODEL_PORT),
            '--host', '127.0.0.1',
            '--served-model-name', str(LOCAL_MODEL_DIR),
            '--trust-remote-code',
            '--dtype', 'bfloat16',
            '--max-model-len', '4096',
            '--gpu-memory-utilization', '0.85',
            '--enforce-eager',
            # 核心: 关掉视觉模态的内存 profiling, 在 ROCm 上规避 vision encoder warmup 挂住
            '--limit-mm-per-prompt.image', '0',
            '--limit-mm-per-prompt.video', '0',
        ]
        log_path = REPO_DIR / 'cache' / 'vllm_serve.log'
        log_path.parent.mkdir(parents=True, exist_ok=True)
        log_f = open(log_path, 'wb')
        print('starting vllm serve (text-only mode, limit-mm-per-prompt=0) ...')
        print(f'  cmd: {" ".join(cmd)}')
        print(f'  logs: {log_path}')
        proc = subprocess.Popen(
            cmd,
            stdout=log_f, stderr=subprocess.STDOUT, env=env,
            preexec_fn=os.setsid,
        )
        atexit.register(lambda: os.killpg(os.getpgid(proc.pid), signal.SIGTERM) if proc.poll() is None else None)

        # vLLM 用 spawn 起 EngineCore 子进程, 父 wrapper 有时会先退出, EngineCore 还在加载权重。
        # 所以 "proc.poll() is not None" 不能直接当失败 —— 必须看端口 / HTTP 是不是真的没起来。
        ready = False
        max_wait = 900  # ROCm 首跑要编 kernel + 加载 17 GiB 权重 + warmup, 给 15 分钟
        start = time.time()
        proc_exited_logged = False
        last_log_size = 0
        while time.time() - start < max_wait:
            # 1) 端口开了 + /v1/models 200 才算 ready
            try:
                with urllib.request.urlopen(f'http://127.0.0.1:{LOCAL_MODEL_PORT}/v1/models', timeout=2) as r:
                    if r.status == 200:
                        ready = True
                        break
            except Exception:
                pass
            # 2) proc 退出只是 warning, 不直接 raise (EngineCore 可能还在跑)
            if proc.poll() is not None and not proc_exited_logged:
                print(f'  note: vllm wrapper 进程已退出 (rc={proc.returncode}), 继续等 EngineCore 子进程把端口开起来')
                proc_exited_logged = True
            # 3) 每 20s 打印 log 尾部 5 行, 让你看到它在干什么
            elapsed = int(time.time() - start)
            if elapsed > 0 and elapsed % 20 == 0:
                try:
                    sz = log_path.stat().st_size
                    if sz != last_log_size:
                        with open(log_path, 'rb') as lf:
                            lf.seek(max(0, sz - 4096))
                            tail = lf.read().decode('utf-8', errors='replace').splitlines()[-5:]
                        print(f'  [{elapsed}s/{max_wait}s] vllm log tail:')
                        for line in tail:
                            print(f'    | {line}')
                        last_log_size = sz
                    else:
                        print(f'  [{elapsed}s/{max_wait}s] still waiting ...')
                except Exception:
                    print(f'  [{elapsed}s/{max_wait}s] still waiting ...')
            time.sleep(2)
        if not ready:
            # 真没起来: 看看 proc 死透了没, log 末尾给出来
            try:
                tail = log_path.read_text('utf-8', errors='replace').splitlines()[-30:]
                print('--- vllm_serve.log tail ---')
                print('\n'.join(tail))
            except Exception:
                pass
            raise RuntimeError(f'vLLM server {max_wait}s 内没起来，看 {log_path}')
        LOCAL_API_PROC = proc
        print('vLLM server ready')

    # 配置变量给后续抽取 cell
    os.environ['LLM_PLATFORM'] = 'custom'
    os.environ['CUSTOM_API_KEY'] = 'EMPTY'
    os.environ['CUSTOM_BASE_URL'] = f'http://127.0.0.1:{LOCAL_MODEL_PORT}/v1'
    os.environ['CUSTOM_MODEL_NAME'] = str(LOCAL_MODEL_DIR)
    LLM_PLATFORM = 'custom'
    print('platform=custom')
    print(f'base_url=http://127.0.0.1:{LOCAL_MODEL_PORT}/v1')
    print(f'model={LOCAL_MODEL_DIR}')
else:
    print('skip (using cloud_api)')


## 3. 抽取原始多轮对话

调用 `extract` 模块（基于 KMnO4-zx/extract-dialogue 改造）。

- **断点续跑**：每个 chunk 抽完会写本地 cache，重跑只补做没完成的 chunk。
- **输出**：一行一条对话 jsonl，`{chunk_id, dialogue_index, role, dialogue}`。
- 抽全本一般 30 分钟 - 2 小时，看小说长度和 LLM 速度。

In [ ]:
import sys
sys.path.insert(0, str(REPO_DIR))

from extract import DialogueExtractor, Config

# 让 extract.Config 用 notebook 设定的目录
Config.CACHE_DIR = str(CACHE_DIR)
Config.CHUNK_SIZE = CHUNK_SIZE_TOKENS
Config.MAX_WORKERS = MAX_WORKERS
Config.set_platform(LLM_PLATFORM)

extractor = DialogueExtractor(
    platform=LLM_PLATFORM,
    max_workers=MAX_WORKERS,
    include_chunk_id=True,
    save_chunk_text=False,
)

assert NOVEL_TXT.exists(), f"小说原文不存在: {NOVEL_TXT}"
print(f"开始抽取: {NOVEL_TXT}")
print(f"输出: {RAW_JSONL}")
extractor.extract_dialogues_concurrent(
    file_path=str(NOVEL_TXT),
    output_file=str(RAW_JSONL),
)
print("抽取完成")

## 4. 速览抽取结果

In [ ]:
import json
from collections import Counter

lines = []
with RAW_JSONL.open(encoding='utf-8') as f:
    for ln in f:
        ln = ln.strip()
        if ln:
            lines.append(json.loads(ln))

print('总对话行数:', len(lines))
role_counts = Counter(l['role'] for l in lines)
print('Top 15 角色发言数:')
for role, n in role_counts.most_common(15):
    mark = '  <-- 目标' if role == TARGET_ROLE else ''
    print(f'  {role}: {n}{mark}')

target_n = role_counts.get(TARGET_ROLE, 0)
if target_n == 0:
    print(f'⚠️ 目标角色 {TARGET_ROLE!r} 一句都没抽到，检查名字是否拼写正确，或者小说里角色称呼是否一致')
elif target_n < 100:
    print(f'⚠️ 目标角色发言只有 {target_n} 条，数据量偏少，建议换更大角色或者更长小说')
else:
    print(f'✓ 目标角色发言 {target_n} 条')

## 5. 转成多轮 ShareGPT SFT 数据

保留每个 chunk 的真实对话流，目标角色每段连续发言作为 `gpt` turn，其他角色作为前置 `human` turn。这样模型学到的就是「跨轮维持人设」，而不是单轮 Q→A 切片。

In [ ]:
import subprocess
cmd = [
    'python3', str(REPO_DIR / 'scripts' / 'build_chat_sft_multiturn.py'),
    '--input', str(RAW_JSONL),
    '--target-role', TARGET_ROLE,
    '--run-name', RUN_NAME,
    '--novel-title', NOVEL_TITLE,
    '--out-dir', str(SFT_DIR),
    '--valid-ratio', str(VALID_RATIO),
    '--seed', str(SEED),
]
if MAX_CONVERSATIONS > 0:
    cmd += ['--max-conversations', str(MAX_CONVERSATIONS)]
print(' '.join(cmd))
subprocess.check_call(cmd)

## 6. 写 dataset_info.json

让 LLaMA Factory 知道训练数据集的列名/格式。`02_train.ipynb` 会读这个文件。

In [ ]:
import json

info_path = SFT_DIR / 'dataset_info.json'
info = {}
if info_path.exists():
    info = json.loads(info_path.read_text(encoding='utf-8'))

for split in ["train", "valid"]:
    key = f"{RUN_NAME}_{split}"
    info[key] = {
        "file_name": f"{RUN_NAME}_chat_{split}.json",
        "formatting": "sharegpt",
        "columns": {"messages": "conversations", "system": "system"},
        "tags": {"role_tag": "from", "content_tag": "value",
                 "user_tag": "human", "assistant_tag": "gpt"},
    }
info_path.write_text(json.dumps(info, ensure_ascii=False, indent=2), encoding="utf-8")
print('updated', info_path)
print('keys:', list(info.keys()))

## 7. 完成

现在 `data/{RUN_NAME}_chat_train.json` 已经就绪。

**下一步**：打开 `02_train.ipynb`，在它的参数区把 `RUN_NAME` 改成和本文件一致，跑训练。